Import Libraries

In [0]:
import pyspark.sql.functions as F

In [0]:
# 1. Îi spunem motorului Spark să citească fișierul CSV
# .option("header", "true") -> Îi zice că primul rând conține numele coloanelor
# .option("inferSchema", "true") -> Îi zice să ghicească singur care sunt numere și care sunt texte

df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Workspace/Users/emanuelbahna@gmail.com/weather_data.csv")

# 2. Afișăm tabelul (Echivalentul la 'Data Preview' din ADF)
df_bronze.show()

+--------------------+-------------------+-------------+--------------+------------+------+------------+------------+
| ingestion_timestamp|temperature_celsius|windspeed_kmh|wind_direction|weather_code|is_day|location_lat|location_lon|
+--------------------+-------------------+-------------+--------------+------------+------+------------+------------+
|2026-02-25 22:51:...|                5.5|          8.7|           318|           2| false|     45.4375|      23.375|
|2026-02-26 07:47:...|                5.5|          0.4|            90|           0|  true|     45.4375|      23.375|
|2026-02-26 07:58:...|                5.5|          0.4|            90|           0|  true|     45.4375|      23.375|
|2026-02-26 07:58:...|                5.5|          0.4|            90|           0|  true|     45.4375|      23.375|
|2026-02-26 07:58:...|                5.5|          0.4|            90|           0|  true|     45.4375|      23.375|
|2026-02-26 07:59:...|                5.5|          0.4|

In [0]:
df_silver = df_bronze.withColumn(
    'ingestion_timestamp', F.to_date('ingestion_timestamp')
).filter(
    F.year('ingestion_timestamp') <= 2030
).withColumn(
    'temperature_fahrenheit', F.col('temperature_celsius') * 1.8 + 32
)


df_silver.show()

+-------------------+-------------------+-------------+--------------+------------+------+------------+------------+----------------------+
|ingestion_timestamp|temperature_celsius|windspeed_kmh|wind_direction|weather_code|is_day|location_lat|location_lon|temperature_fahrenheit|
+-------------------+-------------------+-------------+--------------+------------+------+------------+------------+----------------------+
|         2026-02-25|                5.5|          8.7|           318|           2| false|     45.4375|      23.375|                  41.9|
|         2026-02-26|                5.5|          0.4|            90|           0|  true|     45.4375|      23.375|                  41.9|
|         2026-02-26|                5.5|          0.4|            90|           0|  true|     45.4375|      23.375|                  41.9|
|         2026-02-26|                5.5|          0.4|            90|           0|  true|     45.4375|      23.375|                  41.9|
|         2026-02-26

In [0]:
df_gold = df_silver.groupBy("ingestion_timestamp").agg(
    F.avg(F.col('temperature_celsius')).alias('avg_temperature_celsius'),
    F.max(F.col('windspeed_kmh')).alias('max_windspeed_kmh'),
).withColumns(
   {
       'max_windspeed_kmh':  F.round(F.col('max_windspeed_kmh'), 2),
        'avg_temperature_celsius': F.round(F.col('avg_temperature_celsius'), 1)
   }
)


df_gold.show()      

+-------------------+-----------------------+-----------------+
|ingestion_timestamp|avg_temperature_celsius|max_windspeed_kmh|
+-------------------+-----------------------+-----------------+
|         2026-02-28|                    5.5|              3.2|
|         2026-02-25|                    5.5|              8.7|
|         2026-02-26|                    5.4|              8.3|
+-------------------+-----------------------+-----------------+



In [0]:
df_gold.write \
    .mode('overwrite') \
    .saveAsTable('weather_gold')
    

In [0]:
%sql
SELECT * FROM weather_gold;

ingestion_timestamp,avg_temperature_celsius,max_windspeed_kmh
2026-02-28,5.5,3.2
2026-02-25,5.5,8.7
2026-02-26,5.4,8.3


In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,weather_gold,false
